# Análisis Exploratorio de Datos (EDA)

### Objetivos del notebook:

* Suministrar contexto sobre el problema y la naturaleza de los datos disponibles.
* Evaluar la integridad del dataset y definir estrategias de limpieza/tratamiento.
* Establecer la relación entre las variables independientes y la variable objetivo (`fraude`).
* Identificar y documentar las variables que deben ser descartadas del *pipeline* final debido a
 problemas de calidad o su relación con la variable objetivo.

In [29]:
import os
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

## Configuración del notebook

In [30]:
# Constantes usadas en el notebook
ROOT = "../"
DATA = os.path.join(ROOT, "data")
paleta_color = "Pastel6"

In [32]:
# Lectura del dataset de transacciones
transactions = pd.read_csv(f"{DATA}/dataset.csv", parse_dates=["fecha"])
transactions.head(3)

,a,b,c,d,e,f,g,h,j,k,l,m,n,o,p,fecha,monto,score,fraude
0,4,0.6812,50084.12,50.0,0.000000,20.0,AR,1,cat_d26ab52,0.365475,2479.0,952.0,1,NaN,Y,2020-03-20 09:28:19,57.63,100,0
1,4,0.6694,66005.49,0.0,0.000000,2.0,AR,1,cat_ea962fb,0.612728,2603.0,105.0,1,Y,Y,2020-03-09 13:58:28,40.19,25,0
2,4,0.4718,7059.05,4.0,0.463488,92.0,BR,25,cat_4c2544e,0.651835,2153.0,249.0,1,Y,Y,2020-04-08 12:25:55,5.77,23,0


## Análisis de Calidad de dataset

In [33]:
# 1.1 Información general del dataset
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 19 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   a       150000 non-null  int64         
 1   b       137016 non-null  float64       
 2   c       137016 non-null  float64       
 3   d       149635 non-null  float64       
 4   e       150000 non-null  float64       
 5   f       149989 non-null  float64       
 6   g       149806 non-null  str           
 7   h       150000 non-null  int64         
 8   j       150000 non-null  str           
 9   k       150000 non-null  float64       
 10  l       149989 non-null  float64       
 11  m       149635 non-null  float64       
 12  n       150000 non-null  int64         
 13  o       41143 non-null   str           
 14  p       150000 non-null  str           
 15  fecha   150000 non-null  datetime64[us]
 16  monto   150000 non-null  float64       
 17  score   150000 non-null  int64         


In [34]:
# 1.2 Estadísticas descriptivas del dataset
print("### Estadísticas descriptivas para variables numéricas: ###")
display(transactions.describe(include='number').T.round(2))

print("### Estadísticas descriptivas para variables categóricas: ###")
display(transactions.describe(include='str').T)

### Estadísticas descriptivas para variables numéricas: ###


,count,mean,std,min,25%,50%,75%,max
a,150000.0,3.71,0.75,1.00,4.00,4.00,4.00,4.00
b,137016.0,0.73,0.13,0.00,0.68,0.76,0.81,1.00
c,137016.0,260445.11,846436.14,0.16,9679.91,43711.66,145443.63,13878743.71
d,149635.0,21.68,20.06,0.00,2.00,14.00,50.00,50.00
e,150000.0,0.22,2.43,0.00,0.00,0.10,0.28,833.33
f,149989.0,51.17,709.47,-5.00,1.00,8.00,33.00,145274.00
h,150000.0,14.19,14.16,0.00,3.00,9.00,21.00,58.00
k,150000.0,0.50,0.29,0.00,0.25,0.50,0.75,1.00
l,149989.0,2305.41,1712.38,0.00,910.00,1937.00,3445.00,7544.00
m,149635.0,299.97,321.08,0.00,42.00,193.00,459.00,2225.00


### Estadísticas descriptivas para variables categóricas: ###


,count,unique,top,freq
g,149806,51,BR,111628
j,150000,8324,cat_43b9c10,2331
o,41143,2,Y,24091
p,150000,2,Y,83129


**Observaciones:**
* `f` tiene valores negativos (mínimo -5), que podrían ser códigos especiales. Además su valores máximos estan muy por encima de su mediana (cola larga).
* `d` tiene como máximo 50 y al menos el 25% de los registros en ese valor.
* `c`, `e` también tienen máximos muy lejos de su mediana (colas largas).
* `j` tiene 8,324 categorías y `g` 51 países, con Brasil como el más frecuente.

In [35]:
# 1.3 Cantidad de variables explicativas (sin target ni la fecha)
features = transactions.drop(columns=['fraude', 'fecha'])
print(f"Variables numéricas: {features.select_dtypes(include='number').shape[1]}")
print(f"Variables categóricas: {features.select_dtypes(include='str').shape[1]}")
print(f"Numéricas con pocos valores (posibles categóricas): {[c for c in features.select_dtypes(include='number') if features[c].nunique() <= 10]}")

Variables numéricas: 13
Variables categóricas: 4
Numéricas con pocos valores (posibles categóricas): ['a', 'n']


In [36]:
# Valores únicos de 'a' y 'n' (posibles categóricas)
display(features['a'].value_counts(dropna=False))
display(features['n'].value_counts(dropna=False))

a
4    128579
2     14378
1      4195
3      2848
Name: count, dtype: int64

n
1    135353
0     14647
Name: count, dtype: int64

**Decisión:** 

`a` se trata como una variable categórica nominal y `n` se tratan como una variable binaria.

In [37]:
# 2.1 Distribución de la variable objetivo (target): por cantidad y por monto
resumen_target = transactions.groupby('fraude')['monto'].agg(transacciones='size', monto_total='sum')
resumen_target['% transacciones'] = resumen_target['transacciones'] / resumen_target['transacciones'].sum() * 100
resumen_target['% monto'] = resumen_target['monto_total'] / resumen_target['monto_total'].sum() * 100
resumen_target.round(2)


,transacciones,monto_total,% transacciones,% monto
fraude,,,,
0,142500,5981199.00,95.0,91.62
1,7500,547271.12,5.0,8.38


**Observación:** 

* El 5% de las transacciones son fraude (desbalance de clases), pero representan el 8.4% del monto total, así que los fraudes suelen ser de montos más altos.

**Decisión:** 
* El modelo se evalúa con las siguientes métricas: *recall, precision y false positive rate*. Se excluye el accuracy por el desbalance de clases.
* Se evaluará `class_weight` como alternativa al desbalance de clases, considerando que penaliza más los errores sobre la clase minoritaria `fraude`.


In [38]:
# 3.1 Valores faltantes y tasa de fraude con ellos
tasa_fraude_nulo = {c: transactions.loc[transactions[c].isna(), 'fraude'].mean() * 100 for c in transactions.columns}

null_summary = pd.DataFrame({
    'cantidad faltantes': transactions.isna().sum(),
    'porcentaje faltantes': transactions.isna().mean() * 100,
    'tasa fraude en nulos (%)': pd.Series(tasa_fraude_nulo),
}).round(2)

null_summary = null_summary[null_summary['cantidad faltantes'] > 0].sort_values('cantidad faltantes', ascending=False)
null_summary


,cantidad faltantes,porcentaje faltantes,tasa fraude en nulos (%)
o,108857,72.57,2.05
b,12984,8.66,6.38
c,12984,8.66,6.38
d,365,0.24,7.95
m,365,0.24,7.95
g,194,0.13,7.73
f,11,0.01,0.00
l,11,0.01,0.00


In [39]:
# 3.2 ¿Los nulos de b/c y d/m están en las mismas filas?
for x, y in [('b', 'c'), ('d', 'm')]:
    print(f"'{x}' y '{y}' tienen nulos en las mismas filas: {transactions[x].isna().equals(transactions[y].isna())}")

'b' y 'c' tienen nulos en las mismas filas: True
'd' y 'm' tienen nulos en las mismas filas: True


**Observaciones:** 

* Excepto `o`, todas las variables tienen menos de 9% de nulos. 
* Si bien `o` tiene 72.6% de nulos, el % de fraude de estas transacciones es 2.05% (menor al 5% del dataset). La ausencia del dato es una señal de menor riesgo y por tanto no se descarta la variable por su % de nulos. 
* En `b/c` ocurre lo contrario: el % de fraude en los nulos es mayor (6.38% frente a 5%), sobre 12.984 registros. Que falte el dato es una señal de mayor riesgo.
* En `d/m` y `g` la tasa también es mayor (7.9% y 7.7%), pero con tan pocos registros (menos de 400) no es una señal concluyente.
* Los nulos de `b/c` y `d/m` están en las mismas filas, lo que sugiere que cada par proviene de la misma fuente de datos.
* `f` y `l` tienen solo 11 nulos, muy pocos para concluir algo de su 0% de fraude.

**Decisión:** 

* Teniendo en cuenta que el modelo desplegado debe tener la capacidad de decidir sobre transacciones incompletas y que los registros nulos son informativos (señal con respecto al % fraude) se opta por las siguientes estrategias:
    * Variables numéricas (`b/c`, `d/m`, `f/l`): se conservan los nulos sin imputar. Algoritmos como LightGBM manejan de forma nativa los nulos y pueden aprovechar esta ausencia directamente en sus particiones, conservando la señal detectada.
    *  `o` y `g` (categóricas): el nulo se trata como una categoría más ("sin_registro").

In [40]:
# 4. Valores sospechosos: negativos en numéricas y strings tipo "unknown" en categóricas
num_cols = transactions.select_dtypes(include='number').columns
negativos = (transactions[num_cols] < 0).sum()
print("Columnas con valores negativos:")
print(negativos[negativos > 0])

cat_cols = transactions.select_dtypes(include='str').columns
strings_sospechosos = ['unknown', 'n/a', 'nan', 'null', 'otros', 'other', '', 'na']
for col in cat_cols:
    conteo = transactions[col].str.lower().isin(strings_sospechosos).sum()
    if conteo > 0:
        print(f"'{col}': {conteo} registros sospechosos")


Columnas con valores negativos:
f    1277
dtype: int64


In [41]:
# Verificación de la distribución de valores negativos en la columna 'f'
print("\nDistribución de negativos en 'f':")
print(transactions.loc[transactions['f'] < 0, 'f'].value_counts().sort_index())
print("\nTasa de fraude (%) según f < 0:")
print(transactions.groupby(transactions['f'] < 0)['fraude'].mean().mul(100).round(2))


Distribución de negativos en 'f':
f
-5.0      1
-4.0      6
-3.0     58
-2.0    335
-1.0    877
Name: count, dtype: int64

Tasa de fraude (%) según f < 0:
f
False    4.97
True     8.77
Name: fraude, dtype: float64


**Observaciones:** 

* `f` es la única variable con negativos (1,277 registros, casi todos -1 y -2). Las transacciones con `f < 0` tienen 8.8% de fraude, contra 5.0% en el resto.
* No hay variables categóricas con valores tipo "unknown"

**Decisión:** 

* Sin diccionario de datos no se puede saber si los valores negativos de `f` son errores o códigos especiales. 
* Se conservan sin modificar: al ser una variable numérica, algoritmos basados en arboles pueden aislarlos con un corte en `f < 0` si aportan información.


In [42]:
# 5. Identificación de duplicados exactos (toda la fila es idéntica)
duplicados_exactos = transactions.duplicated().sum()
print(f"Duplicados exactos en el dataset: {duplicados_exactos}")

Duplicados exactos en el dataset: 0


In [43]:
# 6. Cardinalidad por columna
unique_values_summary = pd.DataFrame({
    'tipo de dato': transactions.dtypes,
    'cantidad unicos': transactions.nunique(),
    'porcentaje unicos': transactions.nunique() / len(transactions) * 100,
    'valor más frecuente': transactions.apply(lambda s: s.value_counts().index[0]),
    '% valor más frecuente': transactions.apply(lambda s: s.value_counts(normalize=True).iloc[0] * 100),
}).round(2).sort_values('cantidad unicos', ascending=False)

unique_values_summary


,tipo de dato,cantidad unicos,porcentaje unicos,valor más frecuente,% valor más frecuente
k,float64,150000,100.00,0.365475,0.00
fecha,datetime64[us],145813,97.21,2020-03-19 15:30:23,0.00
c,float64,135090,90.06,5.55,0.01
e,float64,43208,28.81,0.0,43.37
monto,float64,17831,11.89,23.63,0.16
j,str,8324,5.55,cat_43b9c10,1.55
b,float64,7672,5.11,1.0,1.32
l,float64,7297,4.86,0.0,1.35
m,float64,1793,1.20,0.0,10.62
f,float64,1338,0.89,0.0,16.93


In [44]:
# 6.1 Efecto de agrupar categorías poco frecuentes en "otros"
resumen = []
for col in ['g', 'j']:
    conteo = transactions[col].value_counts()
    for umbral in [20, 50, 100]:
        frecuentes = conteo[conteo >= umbral].index
        es_otros = ~transactions[col].isin(frecuentes) & transactions[col].notna()
        resumen.append({
            'columna': col,
            'umbral (mín. registros)': umbral,
            'categorías conservadas': len(frecuentes),
            'filas en OTROS': es_otros.sum(),
            '% filas en OTROS': es_otros.mean() * 100,
            'tasa fraude en OTROS (%)': transactions.loc[es_otros, 'fraude'].mean() * 100,
        })

print(f"Tasa de fraude global: {transactions['fraude'].mean() * 100:.2f}%")
pd.DataFrame(resumen).round(2)


Tasa de fraude global: 5.00%


,columna,umbral (mín. registros),categorías conservadas,filas en OTROS,% filas en OTROS,tasa fraude en OTROS (%)
0,g,20,10,131,0.09,11.45
1,g,50,9,174,0.12,8.62
2,g,100,6,380,0.25,7.11
3,j,20,1400,30951,20.63,3.53
4,j,50,615,55028,36.69,3.87
5,j,100,271,78715,52.48,4.29


**Observaciones:**
* `k` tiene un valor distinto en cada fila, lo que la hace parecer un identificador.
* `e` vale 0 en el 43% de los registros.
* Agrupar las categorías con menos de 20 registros deja el grupo "otros" con el 20.6% de las filas en `j` y menos del 0.3% en `g`. En ambos casos la tasa de fraude de "OTROS" se ubica por encima del 5% global, aunque en `g` el grupo es demasiado pequeño para concluir que la diferencia sea real.

**Decisión:** 
* `k` es candidata a descartar, pendiente de confirmar en el análisis exploratorio. 
* Se usa el umbral de 20 registros para `j` y `g`. La lista de categorías frecuentes se calcula únicamente con los datos de entrenamiento, para no filtrar información del conjunto de evaluación.

In [45]:
# 7. Revisión de la ventana temporal
fecha_min = transactions['fecha'].min()
fecha_max = transactions['fecha'].max()
dias_totales = (fecha_max - fecha_min).days
print(f"Ventana: {fecha_min} a {fecha_max} ({(fecha_max - fecha_min).days} días)")

# Tasa de fraude por semana
semanal = transactions.groupby(transactions['fecha'].dt.to_period('W'))['fraude'].agg(transacciones='size', tasa_fraude='mean')
semanal['tasa_fraude'] = (semanal['tasa_fraude'] * 100).round(2)
semanal

Ventana: 2020-03-08 00:02:15 a 2020-04-21 23:59:56 (44 días)


,transacciones,tasa_fraude
fecha,,
2020-03-02/2020-03-08,2838,4.37
2020-03-09/2020-03-15,24523,4.83
2020-03-16/2020-03-22,25199,4.56
2020-03-23/2020-03-29,20069,5.69
2020-03-30/2020-04-05,17434,5.78
2020-04-06/2020-04-12,24443,5.41
2020-04-13/2020-04-19,25332,4.29
2020-04-20/2020-04-26,10162,4.76


**Observación:** 
* Los datos cubren 44 días (08-mar a 21-abr de 2020). La tasa semanal de fraude varía entre 4.3% y 5.8%. La primera y la última semana están incompletas.

**Decisión:** 
* train, val y test se separan por fecha (entrenar con las semanas más antiguas y evaluar con las más recientes), garantizando que los cortes respeten semanas completas. Así se simula el uso real del modelo, que decide sobre transacciones futuras, y se evita que información del periodo evaluado se filtre al entrenamiento.

### Recomendaciones para el modelado

Las decisiones justificadas arriba se resumen así:

| Aspecto | Decisión |
|---|---|
| Nulos numéricos (`b/c`, `d/m`, `f/l`) | Se conservan sin imputar |
| Nulos categóricos (`o`, `g`) | Categoría "sin_registro" |
| Negativos en `f` | Se conservan sin modificar |
| Alta cardinalidad (`j`, `g`) | Categorías con <20 registros a "otros" |
| `a`, `n` | Categórica nominal y binaria |
| `k` | Candidata a descartar, pendiente de confirmar |
| Partición | Por fecha (train/val/test), respetando semanas completas |
| Métricas | Recall, precision y FPR (se excluye accuracy) |

Y tres decisiones que se derivan del diagnóstico:

* **Modelo: LightGBM.** Maneja los nulos de forma nativa, que es justo lo que este dataset necesita; acepta categóricas sin codificación previa; y no se ve afectado por las colas largas de `c`, `e`, `f` y `monto`.
* **Categóricas (`g`, `o`, `p`, `a`, `j`): se pasan como categóricas nativas**, sin codificación previa.
* **Colas largas: no se recortan ni se transforman**, por la misma razón anterior.


## Análisis exploratorio

In [ ]:
# Seleccionar solo variables numéricas
num_cols = transactions.select_dtypes(include=['int64', 'float64']).columns
target_col = 'fraude'

if target_col in num_cols:
    num_cols = num_cols.drop(target_col)

for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # 1. Histograma (Distribución y sesgo)
    sns.histplot(data=transactions, x=col, hue=target_col, kde=True, bins=50, ax=axes[0], palette='pastel6')
    axes[0].set_title(f'Distribución de {col} (por clase)', size=11)
    axes[0].set_xlabel(col, size=10)
    axes[0].set_ylabel('Frecuencia', size=10)

    # 2. Boxplot (Detección de outliers)
    sns.boxplot(data=transactions, y=col, x=target_col, ax=axes[1], hue=target_col, palette='pastel6')
    axes[1].set_title(f'Boxplot de {col} (por clase)', size=11)
    axes[1].set_xlabel(target_col, size=10)
    axes[1].set_ylabel(col, size=10)

    # Eliminar las leyendas automáticas de cada eje
    for ax in axes:
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()

    # Crear una leyenda única con los valores del target
    target_values = sorted(transactions[target_col].dropna().unique().tolist())
    colors = dict(zip(target_values, sns.color_palette('pastel6', n_colors=len(target_values))))

    handles = [
        plt.Line2D([0], [0], linestyle='None', marker='o', markersize=7, markerfacecolor=colors[val], markeredgecolor='none', label=str(val))
        for val in target_values
    ]

    fig.subplots_adjust(right=0.82)
    fig.legend(handles=handles, title=target_col, loc='center left', bbox_to_anchor=(0.84, 0.5), frameon=False, fontsize=9)

    plt.tight_layout(rect=[0, 0, 0.8, 1])
    plt.show()

In [ ]:
# Seleccionar variables categóricas
cat_cols = transactions.select_dtypes(include=['object', 'category', 'bool']).columns

for col in cat_cols:
    # Evitar graficar variables con demasiadas categorías (alta cardinalidad como IDs)
    if transactions[col].nunique() < 20:
        plt.figure(figsize=(10, 5))
        
        # Gráfico de barras agrupado por el target
        sns.countplot(data=transactions, x=col, hue=target_col, order=transactions[col].value_counts().index)
        
        plt.title(f'Frecuencia de {col} agrupado por {target_col}')
        plt.xticks(rotation=45, ha='right') # Rota las etiquetas si son muy largas
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️ Se omitió '{col}' por tener alta cardinalidad ({transactions[col].nunique()} valores únicos).")